# Load COCO Reference Images and Libraries

In [ ]:
#Author: Jaden Barnwell, 2026
!pip install -q --upgrade huggingface_hub accelerate datasets torch-fidelity torchmetrics pillow tqdm requests
!pip show torch transformers accelerate
!pip install diffusers==0.20.0 transformers==4.35.0 -q
#!pip install diffusers==0.14.0 transformers==4.30.0 -q
!pip install opencv-python-headless -q
!pip install controlnet-aux -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.35.0 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.12.0 which is incompatible.
tokenizers 0.14.1 requires huggingface_hub<0.18,>=0.16.4, but you have huggingface-hub 1.12.0 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Name: torch
Version: 2.4.1+cu124
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3
Location: /usr/local/lib/python3.11/dist-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-curand-cu12, n

In [2]:
import os
import sys

if not os.path.exists('Faster-Diffusion'):
    os.system('git clone https://github.com/hutaiHang/Faster-Diffusion.git')
else:
    print('Faster-Diffusion repo already cloned')

sys.path.insert(0, './Faster-Diffusion')

Faster-Diffusion repo already cloned


In [3]:
import json
import time
import torch
import numpy as np
from tqdm import tqdm
from diffusers import StableDiffusionPipeline, DiffusionPipeline
from torch_fidelity import calculate_metrics
from utils_sd import register_parallel_pipeline, register_faster_forward
from PIL import Image
import torch.nn.functional as F
from transformers import CLIPProcessor, CLIPModel

/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/controlnet_aux/mediapipe_face/mediapipe_face_common.py:7: UserWarning: The module 'mediapipe' is not installed. The package will have limited functionality. Please install it using the command: pip install '

In [4]:
!pip freeze > requirements.txt

In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

BATCH_SIZE = 32
NUM_STEPS = 25
NUM_IMAGES = 1000

OUTPUT_ROOT = "/workspace/outputs"
COCO_DIR = "/workspace/coco_real"
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(COCO_DIR, exist_ok=True)

print(f"Device: {DEVICE}, Batch: {BATCH_SIZE}")

Device: cuda, Batch: 32


In [6]:
import zipfile
import os

if not os.path.exists('val2017'):
    !wget -q http://images.cocodataset.org/zips/val2017.zip
    with zipfile.ZipFile('val2017.zip', 'r') as z:
        z.extractall('.')

if not os.path.exists('annotations'):
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    with zipfile.ZipFile('annotations_trainval2017.zip', 'r') as z:
        z.extractall('.')

print("COCO ready")

COCO ready


In [7]:
import zipfile

!rm -rf val2017
with zipfile.ZipFile('val2017.zip', 'r') as z:
    z.extractall('.')
print("Re-extracted val2017")

Re-extracted val2017


In [8]:
COCO_IMG_DIR = "val2017"
COCO_ANN_FILE = "annotations/captions_val2017.json"


def load_coco_val2017():
    with open(COCO_ANN_FILE, "r") as f:
        data = json.load(f)

    # Map image_id -> captions
    captions_dict = {}
    for ann in data["annotations"]:
        img_id = ann["image_id"]
        captions_dict.setdefault(img_id, []).append(ann["caption"])

    # Map image_id -> filename
    id_to_file = {img["id"]: img["file_name"] for img in data["images"]}

    samples = []
    for img_id, file_name in id_to_file.items():
        path = os.path.join(COCO_IMG_DIR, file_name)
        captions = captions_dict.get(img_id, [])

        if os.path.exists(path) and captions:
            samples.append({
                "image_path": path,
                "caption": captions[0]  # pick first caption
            })

    return samples


samples = load_coco_val2017()
print(f"Loaded {len(samples)} samples")

Loaded 5000 samples


In [9]:
import os
from PIL import Image

OUTPUT_REAL = "/workspace/coco_real"
os.makedirs(OUTPUT_REAL, exist_ok=True)

saved = 0
skipped = 0

for i, sample in enumerate(samples):
    try:
        img = Image.open(sample["image_path"]).convert("RGB")
        img = img.resize((512, 512))
        img.save(f"{OUTPUT_REAL}/{i}.png")
        saved += 1
    except Exception as e:
        print(f"Skipping {sample['image_path']}: {e}")
        skipped += 1

print(f"Saved: {saved}, Skipped: {skipped}")

Saved: 5000, Skipped: 0


In [10]:
!df -h
!pwd
!ls /workspace/

Filesystem                   Size  Used Avail Use% Mounted on
overlay                       40G  1.1G   39G   3% /
tmpfs                         64M     0   64M   0% /dev
mfs#us-wa-1.runpod.net:9421  479T  245T  234T  52% /workspace
shm                          117G     0  117G   0% /dev/shm
/dev/md127                    28T  4.7T   24T  17% /etc/hosts
/dev/nvme1n1p2               1.8T   33G  1.6T   2% /usr/bin/nvidia-smi
tmpfs                       1002G     0 1002G   0% /sys/fs/cgroup
tmpfs                       1002G   12K 1002G   1% /proc/driver/nvidia
tmpfs                       1002G  4.0K 1002G   1% /etc/nvidia/nvidia-application-profiles-rc.d
tmpfs                        201G   64M  201G   1% /run/nvidia-persistenced/socket
tmpfs                       1002G     0 1002G   0% /proc/acpi
tmpfs                       1002G     0 1002G   0% /proc/scsi
tmpfs                       1002G     0 1002G   0% /sys/firmware
tmpfs                       1002G     0 1002G   0% /sys/devices/virtu

### Faster-Diffusion Computation w/ FID Metric

In [11]:
def load_prompts(samples, num_images):
    return [s["caption"] for s in samples[:num_images]]


prompts = load_prompts(samples, NUM_IMAGES)

def load_pipeline(name):
    if name == "sd1.5":
        pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=DTYPE
        )
    elif name == "sd2.1":
        pipe = DiffusionPipeline.from_pretrained(
            "sd2-community/stable-diffusion-2-1",
            torch_dtype=DTYPE
        )

    pipe = pipe.to(DEVICE)
    pipe.set_progress_bar_config(disable=True)
    return pipe

def enable_faster_diffusion(pipe, use_faster):
    if not use_faster:
        return
    register_parallel_pipeline(pipe, mod='50ls')
    pipe.unet.order = 0  # manually initialize order attribute
    register_faster_forward(pipe.unet, mod='50ls')

In [12]:
def generate_images(pipe, prompts, out_dir, use_faster):
    os.makedirs(out_dir, exist_ok=True)

    enable_faster_diffusion(pipe, use_faster)

    all_times = []

    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()

    for i in tqdm(range(0, len(prompts), BATCH_SIZE)):
        batch_prompts = prompts[i:i+BATCH_SIZE]
        batch_size = len(batch_prompts)

        # SAME LATENTS for fairness
        latents = torch.randn(
            (batch_size, pipe.unet.in_channels, 64, 64),
            device=DEVICE,
            dtype=DTYPE
        )

        start = time.time()

        images = pipe(
            batch_prompts,
            num_inference_steps=NUM_STEPS,
            latents=latents
        ).images

        end = time.time()
        all_times.append(end - start)

        for j, img in enumerate(images):
            img.save(f"{out_dir}/{i+j}.png")

    peak_mem = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else 0.0

    return np.mean(all_times), np.std(all_times), peak_mem

def compute_fid(real_dir, fake_dir):
    metrics = calculate_metrics(
        input1=real_dir,
        input2=fake_dir,
        fid=True,
        isc=False,
        kid=False,
        cuda=(DEVICE == "cuda")
    )
    return metrics["frechet_inception_distance"]

In [13]:
NUM_IMAGES = 1000
prompts = load_prompts(samples, NUM_IMAGES)
print(f"Prompts: {len(prompts)}, Batches: {len(prompts) // BATCH_SIZE}")

Prompts: 1000, Batches: 31


In [14]:
configs = [
    ("sd1.5", True),
    ("sd2.1", True),
]

results = {}

for model, faster in configs:
    print(f"\nRunning {model} | faster={faster}")

    pipe = load_pipeline(model)

    out_dir = f"{OUTPUT_ROOT}/{model}_faster"

    mean_t, std_t, peak_mem = generate_images(pipe, prompts, out_dir, faster)

    results[(model, faster)] = {
        "dir": out_dir,
        "time_mean": mean_t,
        "time_std": std_t,
        "peak_mem_gb": peak_mem
    }

    del pipe
    torch.cuda.empty_cache()


Running sd1.5 | faster=True


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["bos_token_id"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["eos_token_id"]` will be overriden.
  0%|          | 0/32 [00:00<?, ?it/s]/tmp/ipykernel_1801/1787877852.py:17: FutureWarning: Accessing config attribute `in_channels` directly via 'UNet2DConditionModel' object attribute is deprecated. Please access 'in_channels' over 'UNet2DConditionModel's config object instead, e.g. 'unet.config.in_channels'.
  (batch_size, pipe.unet.in_channels, 64, 64),
100%|██████████| 32/32 [08:19<00:00, 15.61s/it]



Running sd2.1 | faster=True


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

100%|██████████| 32/32 [07:18<00:00, 13.71s/it]


In [15]:
fid_results = {}

for model in ["sd1.5", "sd2.1"]:
    faster_dir = results[(model, True)]["dir"]
    fid_faster = compute_fid(COCO_DIR, faster_dir)
    fid_results[model] = {"faster": fid_faster}

print("\n=== FINAL RESULTS ===")

for (model, faster), data in results.items():
    print(f"{model} | faster={faster}")
    print(f"  Time:      {data['time_mean']:.3f} ± {data['time_std']:.3f} s")
    print(f"  Peak VRAM: {data['peak_mem_gb']:.2f} GB")

for model, vals in fid_results.items():
    print(f"\n{model} FID:")
    print(f"  Faster: {vals['faster']:.3f}")

Creating feature extractor "inception-v3-compat" with features ['2048']
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:00<00:00, 110MB/s] 
Extracting statistics from input 1
Looking for samples non-recursivelty in "/workspace/coco_real" with extensions png,jpg,jpeg
Found 5000 samples
Processing samples                                                           
Extracting statistics from input 2
Looking for samples non-recursivelty in "/workspace/outputs/sd1.5_faster" with extensions png,jpg,jpeg
Found 1000 samples
Processing samples                                                          
Frechet Inception Distance: 53.3588
Creating feature extractor "inception-v3-compat" with features ['2048']
Extracting statistics from input 1
Looking for samples non-recursivelty in "/workspace/coco_real" with ext


=== FINAL RESULTS ===
sd1.5 | faster=True
  Time:      12.768 ± 1.657 s
  Peak VRAM: 20.84 GB
sd2.1 | faster=True
  Time:      10.742 ± 1.385 s
  Peak VRAM: 20.67 GB

sd1.5 FID:
  Faster: 53.359

sd2.1 FID:
  Faster: 58.772


Frechet Inception Distance: 58.77231


### CLIP Computation

In [16]:
def load_clip():
    model = CLIPModel.from_pretrained(
        "openai/clip-vit-base-patch32"
    ).to(DEVICE)

    processor = CLIPProcessor.from_pretrained(
        "openai/clip-vit-base-patch32"
    )

    model.eval()
    return model, processor

In [17]:
def compute_clip_score(image_dir, prompts, model, processor):
    image_files = sorted(os.listdir(image_dir))

    scores = []

    for i in tqdm(range(0, len(image_files), BATCH_SIZE)):
        batch_files = image_files[i:i+BATCH_SIZE]
        batch_images = []
        batch_texts = []

        for j, file in enumerate(batch_files):
            img = Image.open(os.path.join(image_dir, file)).convert("RGB")
            batch_images.append(img)

            # match prompt index
            idx = i + j
            batch_texts.append(prompts[idx])

        inputs = processor(
            text=batch_texts,
            images=batch_images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=77
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model(**inputs)

            image_embeds = outputs.image_embeds
            text_embeds = outputs.text_embeds

            # Normalize (CRITICAL)
            image_embeds = F.normalize(image_embeds, dim=-1)
            text_embeds = F.normalize(text_embeds, dim=-1)

            # cosine similarity
            batch_scores = (image_embeds * text_embeds).sum(dim=-1)

        scores.extend(batch_scores.cpu().numpy())

    return float(np.mean(scores))

In [18]:
clip_model, clip_processor = load_clip()

clip_results = {}

for model in ["sd1.5", "sd2.1"]:
    faster_dir = results[(model, True)]["dir"]

    print(f"\nComputing CLIP for {model}...")

    clip_faster = compute_clip_score(
        faster_dir, prompts, clip_model, clip_processor
    )

    clip_results[model] = {"faster": clip_faster}

/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:484: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(checkpoint_file, map_location=map


Computing CLIP for sd1.5...


100%|██████████| 32/32 [00:23<00:00,  1.35it/s]



Computing CLIP for sd2.1...


100%|██████████| 32/32 [00:24<00:00,  1.30it/s]


# Results

In [19]:
for model, vals in clip_results.items():
    print(f"\n{model} CLIP:")
    print(f"  Faster: {vals['faster']:.4f}")


sd1.5 CLIP:
  Faster: 0.1464

sd2.1 CLIP:
  Faster: 0.1525


In [ ]:
| Model                               | Mean Time/batch | Time/image | Var/image | Peak VRAM | FID    | CLIP   |
| ----------------------------------- | --------------- | ---------- | --------- | --------- | ------ | ------ |
| SD1.5 + Faster-Diffusion (mod=50ls) | 12.768s         | 0.399s     | 0.052s    | 20.84 GB  | 53.359 | 0.1464 |
| SD2.1 + Faster-Diffusion (mod=50ls) | 10.742s         | 0.336s     | 0.043s    | 20.67 GB  | 58.772 | 0.1525 |